In [ ]:
# Import necessary packages (might need to download some in the terminal)
import os
import re
import pandas as pd
import nltk
import pdfplumber

In [ ]:
# Define input directory containing CAAR market report files
DATA_DIR = "/Users/emilymoore/Downloads/DS 4002 Project 1/caar_market_reports"
# Define output path for the structured dashboard metrics CSV
OUTPUT_CSV = "/Users/emilymoore/Downloads/DS 4002 Project 1/caar_market_reports/caar_2025_dashboard_metrics.csv"

# Create a local folder named "data" if it does not already exist
# exist_ok=True prevents an error if the folder already exists
os.makedirs("data", exist_ok=True) 

# Define regex patterns to extract structured dashboard metrics
# Each key represents a variable name in the dataset
# Each value is a regular expression pattern that captures the numeric value
patterns = {
    "sales": r"([\d,]+)\s+Sales",
    "pending_sales": r"([\d,]+)\s+Pending Sales",
    "new_listings": r"([\d,]+)\s+New Listings",
    "median_list_price": r"\$([\d,]+)\s+Median List Price",
    "median_sales_price": r"\$([\d,]+)\s+Median Sales Price",
    "median_price_per_sqft": r"\$([\d,]+)\s+Median Price Per Square Foot",
    "sold_dollar_volume(millions)": r"\$([\d.]+)\s+Sold Dollar Volume",
    "median_sold/ask_price_ratio": r"([\d.]+%)\s+Median Sold/Ask Price Ratio",
    "median_days_on_market": r"([\d,]+)\s+Median Days on Market",
    "new_construction_sales": r"([\d,]+)\s+New Construction Sales"
}

rows = [] # Initialize an empty list to store extracted rows

# Loop through every file in the CAAR reports directory
for file in os.listdir(DATA_DIR):
    file_path = os.path.join(DATA_DIR, file)
    full_text = ""

    # If file is a PDF, extract text using pdfplumber
    if file.lower().endswith(".pdf"):
        with pdfplumber.open(file_path) as pdf:
            full_text = " ".join(page.extract_text() or "" for page in pdf.pages)
    # If file is a TXT file, read directly        
    elif file.lower().endswith(".txt"):
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            full_text = f.read()
            
    else:
        continue # Skip files that are neither PDF nor TXT

    # Create a dictionary for this file’s extracted metrics
    row = {"file_name": file}

    # Search for each metric using its regex pattern
    for field, pattern in patterns.items():
        match = re.search(pattern, full_text, re.IGNORECASE)
        if match:
            value = match.group(1).replace(",", "") # Remove commas from numeric values
            row[field] = value
        else:
            row[field] = None # If metric not found in file, store as None

    rows.append(row) # Append the extracted row to the list

df_caar = pd.DataFrame(rows) # Convert list of dictionaries into a structured DataFrame
df_caar.to_csv(OUTPUT_CSV, index=False) # Save the extracted market metrics to CSV for analysis

print(f"Saved the reports to {OUTPUT_CSV}")

In [ ]:
# Load the processed meeting minutes dataset
df_mins_with_score = pd.read_csv('/Users/emilymoore/Downloads/DS 4002 Project 1/meeting_mins/housing_uncertainty_results.csv')

# Extract the month/year from file_name for chronological sorting
def extract_date(filename):
    match = re.search(r'(\d{2})-(\d{4})', filename)
    return f"{match.group(2)}-{match.group(1)}" if match else None

df_mins_with_score['month_year'] = df_mins_with_score['file_name'].apply(extract_date) # Apply date extraction function

# Calculate the Uncertainty Score
# Density of housing-uncertainty sentences per housing mention
df_mins_with_score['uncertainty_score'] = df_mins_with_score['housing_uncertainty_sentences'] / df_mins_with_score['housing_mentions']

# Handle cases where housing_mentions is 0 to avoid errors
df_mins_with_score['uncertainty_score'] = df_mins_with_score['uncertainty_score'].fillna(0)

# Sort by date and save
df_mins_with_score = df_mins_with_score.sort_values('month_year')
df_mins_with_score.to_csv('/Users/emilymoore/Downloads/DS 4002 Project 1/meeting_mins/housing_uncertainty_results_with_score.csv', index=False)

print(f'Updated the CSV to {df_mins_with_score}')